1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.

`MERGE INTO` is used to **upsert** data into the Silver customers table.

Upsert means:

* If the customer already exists → **UPDATE** the existing record.
> * If the customer is new → **INSERT** the new record.

We match the records using `customer_id`.

### Example

Suppose our Silver table is:

| customer_id | customer_name | city   |
| ----------- | ------------- | ------ |
| 1           | Rahul         | Delhi  |
| 2           | Amit          | Jaipur |

And the changed customer batch contains:

| customer_id | customer_name | city   |
| ----------- | ------------- | ------ |
| 1           | Rahul         | Mumbai |
| 3           | Neha          | Pune   |

Here:

* Customer `1` already exists, so we **update** the city.
* Customer `3` does not exist, so we **insert** it.

### MERGE Query

```sql
MERGE INTO dev.silver.customers AS target

USING customers_source AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```

### How it works

`ON target.customer_id = source.customer_id`

This checks whether the customer already exists.

`WHEN MATCHED THEN UPDATE SET *`

If the customer exists, update the target record with the source record.

`WHEN NOT MATCHED THEN INSERT *`

If the customer does not exist, insert the new record.

So, `MERGE INTO` allows us to handle **both changed customers and new customers in one operation**.


In [0]:
-- Upsert changed customer records from bronze to silver
MERGE INTO dev.silver.customers AS target
USING dev.bronze.customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity
Catalog permissions.

-->>
The goal is to give different levels of access to two groups:

- **Group 1 (`customer_full_access`)** → Full `SELECT` access to the customer table.
- **Group 2 (`customer_limited_access`)** → Access only to a masked view.

### Create a Masked View

```sql
CREATE OR REPLACE VIEW dev.gold.customers_secure AS
SELECT
    customer_id,
    first_name,
    last_name,
    city,
    state,
    customer_segment,
    customer_status,

    CASE
        WHEN is_account_group_member('customer_full_access')
        THEN email
        ELSE '****MASKED****'
    END AS email,

    CASE
        WHEN is_account_group_member('customer_full_access')
        THEN phone
        ELSE NULL
    END AS phone

FROM dev.gold.customers;


In [0]:
GRANT USE CATALOG ON CATALOG dev
TO `customer_full_access`;

GRANT USE SCHEMA ON SCHEMA dev.gold
TO `customer_full_access`;

GRANT SELECT ON TABLE dev.gold.customers
TO `customer_full_access`;

3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms,
what a DBU is billing for.

-->>
I checked the DBU consumption for the compute resource I used in Databricks.
**DBU (Databricks Unit)** is a unit used by Databricks to measure the amount of compute processing being used.

In simple terms, a DBU represents **how much Databricks compute capacity is being consumed over time**.

DBU consumption depends on factors such as:

- Type of compute resource
- Compute size
- Workload being executed
- Amount of time the compute is running
- Databricks compute configuration

For example, if a cluster uses **2 DBUs per hour** and runs for **3 hours**, the consumption would be approximately:

**2 × 3 = 6 DBUs**

DBUs are used as part of Databricks usage for billing. The actual cost depends on the **DBU rate for the specific compute type and Databricks pricing plan**, along with any underlying cloud infrastructure costs.

### In Plain Terms

A DBU is basically a way for Databricks to measure **how much compute power I am using and for how long**.
So than Data bricks calculate the cost of the servises and infrastructure.

For example, if a account uses **500 DBUs** in a month and price is **0.60 USD Per DBU** the cost would be approximately:

**500 × 0.60 = 300 USD **

##The personal Account usage

To check the personal Account usage we can check the table system.billing.usage to get the total usage

In [0]:
select * from system.billing.usage;